# Multi-Output Linear Regression on the Linnerud Dataset

This notebook practices **multi-output linear regression** end to end and — just as importantly — shows what it looks like when a linear model *does not* fit the data.

The [**Linnerud**](https://scikit-learn.org/stable/datasets/toy_dataset.html#linnerrud-dataset) dataset (built into scikit-learn, no download) records 20 middle-aged men at a fitness club:

- **Features (`X`)** — three *exercise* measurements: `Chins`, `Situps`, `Jumps`.
- **Targets (`y`)** — three *physiological* measurements: `Weight`, `Waist`, `Pulse`.

One `LinearRegression` fits **all three targets at once** (multi-output). We will:

1. **Load & inspect** the data.
2. **Explore** it visually (pairplot + a colored scatter).
3. **Train** a linear regression on a train/test split.
4. **Predict** on held-out data and line predictions up against the truth.
5. **Evaluate** per target with $R^2$ and RMSE.

**Spoiler / lesson:** with only 20 samples and exercise reps that barely relate to body measurements, the model generalizes *badly* — the test $R^2$ values come out **negative**. That is a realistic and useful outcome: it teaches how to read the metrics and recognize when linear regression is simply the wrong tool for the data.

## 0. Setup

All imports collected up front: `numpy`/`pandas` for data, `matplotlib`/`seaborn` for plots, and the scikit-learn pieces for the dataset, the split, the model, and the metrics.

In [ ]:
import numpy as np                                   # numeric arrays + sqrt for RMSE
import pandas as pd                                  # DataFrames for tabular data
import matplotlib.pyplot as plt                      # low-level plotting
import seaborn as sns                                # high-level statistical plots (pairplot)

from sklearn.datasets import load_linnerud           # the built-in toy dataset (20 rows)
from sklearn.model_selection import train_test_split # hold out part of the data for testing
from sklearn.linear_model import LinearRegression    # ordinary least-squares regression
from sklearn.metrics import mean_squared_error, r2_score  # regression evaluation metrics

## 1. Load the data

`as_frame=True` returns the features and targets as pandas objects (rather than plain NumPy arrays), which keeps the column names around so plots and metric tables stay readable.

- `X` = `linner.data`  → shape `(20, 3)`: the exercise features.
- `y` = `linner.target` → shape `(20, 3)`: the physiological targets.

In [ ]:
# Load the Linnerud dataset as pandas objects (features + targets carry column names).
linner = load_linnerud(as_frame=True)

# X: exercise features (Chins, Situps, Jumps).  y: physiological targets (Weight, Waist, Pulse).
# Both are DataFrames of shape (20, 3) because as_frame=True.
x, y = linner.data, linner.target

# Rebuild a features-only DataFrame explicitly from the raw data + feature names.
# (x already is this DataFrame; df is a clearly-named copy we use for the EDA plots below.)
df = pd.DataFrame(data=linner.data, columns=linner.feature_names)

print("features (X):", x.shape, "->", list(x.columns))
print("targets  (y):", y.shape, "->", list(y.columns))

### Peek at the feature rows

A quick `head()` to see the scale of each feature — `Situps` and `Jumps` are in the hundreds while `Chins` is in single/low double digits. Linear regression does not *require* scaling to be correct, but the mismatched scales are worth noticing (they show up again in the coefficients).

In [ ]:
# First five rows of the exercise features.
df.head()

## 2. Explore the features (pairplot)

A seaborn **pairplot** shows every feature against every other feature (scatter plots off the diagonal) and each feature's own distribution (a smooth KDE on the diagonal). It is a fast way to spot correlations, clusters, or outliers before modeling.

`sns.pairplot` builds and manages its *own* figure grid, so we just call it and then `plt.show()`.

In [ ]:
# Pairwise scatter matrix of the three features; diagonal = per-feature density estimate.
# pairplot creates its own Figure/Axes grid internally, so no plt.figure() is needed here.
sns.pairplot(df, diag_kind='kde')
plt.show()

## 3. Feature scatter colored by a target

To bring a target into the picture, we scatter two features (`Chins` on x, `Jumps` on y) and color each point by its `Weight` target. If `Weight` were strongly related to these features we would see a smooth color gradient across the plot; a jumbled mix of colors is an early hint that these features do not explain `Weight` well.

In [ ]:
# Give this plot its own figure so it never draws onto a leftover axis.
plt.figure(figsize=(7, 5))

# x-axis = feature col 0 (Chins), y-axis = feature col 2 (Jumps).
# c = the Weight target -> point color; viridis maps low->dark, high->yellow.
scatter = plt.scatter(x.iloc[:, 0], x.iloc[:, 2], c=y["Weight"],
                      cmap='viridis', edgecolor='k', alpha=0.7)

plt.xlabel(x.columns[0])                    # 'Chins'
plt.ylabel(x.columns[2])                    # 'Jumps'
plt.colorbar(scatter, label='Weight')       # legend for the color = target value
plt.title('Chins vs Jumps, colored by Weight')
plt.show()

## 4. Train / test split and fit

We hold out **20%** of the rows for testing so we can measure how the model does on data it never saw during training. `random_state=42` makes the split reproducible.

A single `LinearRegression` is **multi-output**: fitting it on a target matrix `y` of shape `(n, 3)` learns one independent linear model per target column. For each target it finds coefficients $w$ and intercept $b$ minimizing the squared error

$$\hat{y} = Xw + b, \qquad \min_{w,b} \sum_i (\hat{y}_i - y_i)^2.$$

> Note: with only 20 samples, a 20% test set is just **4 rows** — tiny, so the metrics later are noisy. That small size is itself part of why the model struggles.

In [ ]:
# Split rows into 80% train / 20% test. With 20 rows that is 16 train, 4 test.
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.20, random_state=42
)

# One estimator, fit on all three target columns at once (multi-output regression).
model = LinearRegression()
model.fit(x_train, y_train)   # learns a (3 features -> 3 targets) linear mapping

# coef_ is (n_targets, n_features) = (3, 3); intercept_ is one bias per target.
print("coef_ shape:", model.coef_.shape)
print("intercepts :", np.round(model.intercept_, 3))

## 5. Predict and compare to the truth

`model.predict` returns an array of shape `(n_test, 3)` — one predicted value per target for each test row. We wrap it back into a DataFrame (reusing the test targets' column names and index) so predicted and actual rows line up and are easy to read side by side.

In [ ]:
# Predict all three targets for the held-out test rows -> shape (n_test, 3).
preds = model.predict(x_test)

# Rewrap as a DataFrame with the same columns/index as y_test so rows align 1:1.
pred_df = pd.DataFrame(preds, columns=y_test.columns, index=y_test.index)

print("Predicted:")
print(pred_df.head())
print("\nActual:")
print(y_test.head())

## 6. Evaluate per target

The three targets live on completely different scales (`Weight` ~ 150–200, `Pulse` ~ 50–60), so averaging their errors into one number would be misleading. We pass `multioutput="raw_values"` to get a **separate metric per target**:

- **$R^2$ (coefficient of determination)** — fraction of the target's variance the model explains. $R^2 = 1$ is perfect; $R^2 = 0$ is no better than always predicting the mean; **$R^2 < 0$ means the model is *worse* than just predicting the mean.**
- **RMSE (root mean squared error)** — typical error size, in the target's own units.

$$R^2 = 1 - \frac{\sum_i (y_i - \hat{y}_i)^2}{\sum_i (y_i - \bar{y})^2}, \qquad \text{RMSE} = \sqrt{\tfrac{1}{n}\sum_i (y_i - \hat{y}_i)^2}.$$

In [ ]:
# multioutput="raw_values" -> one score per target instead of an averaged scalar.
r2 = r2_score(y_test, preds, multioutput="raw_values")

# mean_squared_error gives per-target MSE; take the sqrt to get RMSE in each target's units.
rmse = np.sqrt(mean_squared_error(y_test, preds, multioutput="raw_values"))

# Print a small aligned table: target name, its R^2, and its RMSE.
for name, r, e in zip(y_test.columns, r2, rmse):
    print(f"{name:<8} R^2: {r:7.3f}   RMSE: {e:7.3f}")

## 7. Takeaways

- **The workflow is the point.** We loaded data, explored it, split it, fit a *single* multi-output `LinearRegression`, predicted, and evaluated **per target** — a template that transfers to almost any supervised problem.
- **Negative $R^2$ is a real result, not a bug.** All three targets score below 0, meaning the fitted line does worse on the 4 test rows than a constant "always predict the training mean" baseline. Two forces cause this:
  1. **Too little data** — 16 training rows and 3 features is barely enough to estimate the line, and a 4-row test set makes every metric extremely noisy.
  2. **Weak relationship** — exercise reps (chins/situps/jumps) simply do not linearly predict body measurements (weight/waist/pulse) here, exactly as the jumbled colors in the scatter hinted.
- **What to try next:** gather more data, engineer/scale features, add regularization (Ridge/Lasso), or use cross-validation instead of a single tiny split so the score is not dominated by which 4 rows happened to land in the test set.